# LearningCenter: Public RAG Walkthrough

## Retrieval Augmented Generation (RAG) and a Simple Agent Workflow

### This notebook provides a compact, public-facing introduction to the ideas behind RAG in the context of the RADIANT_LLM repository

We will walk through:
1. loading a PDF knowledge source,
2. extracting and chunking text,
3. creating embeddings,
4. running similarity search with FAISS,
5. generating an answer with retrieved context, and
6. wrapping the RAG pipeline inside a simple LangChain agent.

The default document used here is the repository's own supplementary material PDF so readers can explore the same project context discussed in `RADIANT_LLM`.

## Tutorial Setup

- This notebook either downloads a public PDF from the `RADIANT_LLM` repository or uploads PDFs from a local folder.
- You can replace the sample URL with your own public PDF source once you understand the workflow.
- An OpenAI API key is required because the notebook uses hosted embedding and chat models.

## Basic Definitions

- A **retrieval-augmented generation (RAG)** system first finds relevant text, then asks the LLM to answer using that retrieved context.
- A **document chunk** is a short section of a larger document. We split long PDFs into chunks because embedding models and LLMs work better on smaller, focused pieces of text.
- An **embedding** is a numeric vector that represents meaning. Similar text should have vectors that lie close together in the embedding space.
- A **vector store** is a data structure that stores many embeddings so we can search for the nearest ones efficiently.
- **Similarity search** means: given a user query, embed the query and retrieve the chunks whose vectors are closest to the query vector.

# Install Requirements

In [ ]:
# In Colab
# !pip install -U pymupdf faiss-cpu wikipedia openai langchain-openai langchain-experimental langchain langchain-community langchain-core "requests==2.32.4"

# In VS Code
# %pip install -q pymupdf faiss-cpu wikipedia openai langchain langchain-openai langchain-experimental "requests==2.32.4"
%pip install -U pymupdf faiss-cpu wikipedia openai langchain-openai langchain-experimental langchain langchain-community langchain-core "requests==2.32.4"

## Load Dependencies and Initialize the Models

The notebook uses OpenAI for embeddings and response generation, plus FAISS for local vector search.

In [ ]:
import os
import re
import sys
import json
import warnings
import requests
import numpy as np
import fitz  # PyMuPDF
import faiss

from pathlib import Path
from urllib.parse import urlparse
from getpass import getpass

from IPython.display import display, Markdown
from openai import OpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_core.prompts import MessagesPlaceholder
from langchain.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

try:
    from langchain.agents import create_tool_calling_agent
except ImportError:
    from langchain.agents.tool_calling_agent.base import create_tool_calling_agent

try:
    from langchain.agents import AgentExecutor
except ImportError:
    from langchain.agents.agent.base import AgentExecutor

warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

IN_COLAB = "google.colab" in sys.modules
openai_key = os.getenv("OPENAI_API_KEY")
langchain_key = os.getenv("LANGCHAIN_API_KEY")

if IN_COLAB and not openai_key:
    try:
        from google.colab import userdata
        openai_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        openai_key = None

if not openai_key:
    openai_key = getpass("Enter your OPENAI_API_KEY: ")
    os.environ["OPENAI_API_KEY"] = openai_key

gpt_model = "gpt-5.2"
openai_embedding_model = "text-embedding-3-small"

my_llm = ChatOpenAI(
    temperature=0,
    model=gpt_model,
    api_key=openai_key,
    stream_usage=True,
)
my_embeddings = OpenAIEmbeddings(
    model=openai_embedding_model,
    openai_api_key=openai_key,
)
openai_client = OpenAI(api_key=openai_key)

print(f"Running in Colab: {IN_COLAB}")
print(f"Initialized OpenAI model: {gpt_model}")

## Optional: LangSmith Tracing

Keep this off unless you want to trace the workflow while experimenting.

In [ ]:
# Uncomment this block if you want LangSmith tracing.
#
# if langchain_key:
#     os.environ["LANGCHAIN_TRACING_V2"] = "true"
#     os.environ["LANGCHAIN_API_KEY"] = langchain_key
#     print("LangSmith tracing is enabled.")
# else:
#     print("LANGCHAIN_API_KEY was not found. Skipping LangSmith tracing.")

## Load the Repository PDF from GitHub

This notebook downloads the public `radiant-llm-supplementary-material.pdf` file from the `RADIANT_LLM` repository into a runtime-local folder.

Repository PDF link: https://github.com/SmartLabNuclear/RADIANT_LLM/blob/main/radiant-llm-evaluation/radiant-llm-supplementary-material.pdf

If you want to experiment with a different document set later, replace the URL in the next code cell with other public PDF links.

### NB: If you are running locally, the file will be stored under `./class_pdf_library`. In Colab, it will be stored under `/content/class_pdf_library`.


In [ ]:
pdf_directory = Path("/content/class_pdf_library") if IN_COLAB else Path("./class_pdf_library")
pdf_directory.mkdir(parents=True, exist_ok=True)

pdf_urls = [
    # Raw GitHub URL for programmatic download of the repository PDF.
    "https://raw.githubusercontent.com/SmartLabNuclear/RADIANT_LLM/main/radiant-llm-evaluation/radiant-llm-supplementary-material.pdf",
]

def download_pdf(url, save_directory):
    filename = Path(urlparse(url).path).name
    if not filename.lower().endswith(".pdf"):
        filename = filename + ".pdf"

    file_path = save_directory / filename

    if file_path.exists():
        print(f"Already downloaded: {file_path.name}")
        return file_path

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    print(f"Downloaded: {file_path.name}")
    return file_path

downloaded_pdfs = []
for url in pdf_urls:
    downloaded_pdfs.append(download_pdf(url, pdf_directory))

print("\nPDF directory:", pdf_directory.resolve())
print("PDF files:")
for pdf_path in sorted(pdf_directory.glob("*.pdf")):
    print(" -", pdf_path.name)

## Alternative: Upload PDFs from Your Computer

If you want to use your own local files instead of the repo PDF, run the next cell in Google Colab and upload one or more PDF files from your computer.

If you run both the download cell and the upload cell, the notebook will process all PDFs currently stored in `pdf_directory`.

In [ ]:
if IN_COLAB:
    from google.colab import files
    from pathlib import Path

    # Ensure pdf_directory is defined for the upload process
    pdf_directory = Path("/content/class_pdf_library")
    pdf_directory.mkdir(parents=True, exist_ok=True)

    uploaded = files.upload()
    uploaded_pdf_paths = []

    for filename, content in uploaded.items():
        if not filename.lower().endswith(".pdf"):
            print(f"Skipping non-PDF file: {filename}")
            continue

        file_path = pdf_directory / Path(filename).name
        with open(file_path, "wb") as f:
            f.write(content)

        uploaded_pdf_paths.append(file_path)
        print(f"Saved: {file_path.name}")

    if uploaded_pdf_paths:
        print("\nCurrent PDF files in the notebook folder:")
        for pdf_path in sorted(pdf_directory.glob("*.pdf")):
            print(" -", pdf_path.name)
    else:
        print("No PDF files were uploaded.")
else:
    print("This upload cell is intended for Google Colab.")

# PART 1: Retrieval

## Process the PDFs and Extract Text

## Chunking

- **Chunking** means splitting a long document into smaller windows of text.
- If a chunk is too large, it may mix unrelated topics and reduce retrieval quality.
- If a chunk is too small, we may lose context needed to answer the question.
- We often use **overlap** between neighboring chunks so important sentences near a boundary are not lost.

A simple chunking rule is:

$$
\text{next\_start} = \text{current\_start} + \text{chunk\_size} - \text{overlap}
$$

This notebook uses word-based chunking for clarity. In production systems, token-based chunking is also common.

In [ ]:
# Function to extract text from a PDF
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        text += page.get_text()
    return text

# Function to split text into overlapping chunks
def split_text_into_chunks(text, max_chunk_size=1000, overlap_size=50):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + max_chunk_size, len(words))
        chunk = words[start:end]
        chunks.append(" ".join(chunk))
        start += max_chunk_size - overlap_size

    return chunks

# Function to generate embeddings
def get_embedding(text):
    response = openai_client.embeddings.create(
        model=openai_embedding_model,
        input=text,
    )
    embedding = response.data[0].embedding
    return embedding

In [ ]:
# Lists to hold all chunks and their embeddings
all_chunks = []
all_embeddings = []
processed_pdf_count = 0

# Process each PDF in the directory
for pdf_file in sorted(pdf_directory.glob("*.pdf")):
    pdf_path = str(pdf_file)
    print(f"Now Processing:\n {pdf_path}")

    text = extract_text_from_pdf(pdf_path)
    chunks = split_text_into_chunks(text, max_chunk_size=1000, overlap_size=50)

    all_chunks.extend(chunks)

    for chunk in chunks:
        embedding = get_embedding(chunk)
        all_embeddings.append(embedding)

    processed_pdf_count += 1

embedding_matrix = np.array(all_embeddings).astype("float32")

embedding_file_path = pdf_directory / "all_embeddings.npy"
chunks_file_path = pdf_directory / "all_chunks.txt"

np.save(embedding_file_path, embedding_matrix)

with open(chunks_file_path, "w", encoding="utf-8") as f:
    for chunk in all_chunks:
        f.write(chunk + "\n")

print(f"\nProcessed {len(all_chunks)} chunks from {processed_pdf_count} PDFs.")
print(f"Embeddings saved at: {embedding_file_path}")
print(f"Chunks saved at: {chunks_file_path}")

## Perform Similarity Search on a User Query

## Embeddings, Vector Stores, and Similarity Search

- An embedding maps text to a vector in a high-dimensional space:

$$
x \longrightarrow \mathbf{e}(x) \in \mathbb{R}^{d}
$$

- Here, $d$ is the embedding dimension.
- If two texts are semantically similar, their embeddings should be near each other.
- A **vector store** keeps many embeddings together so we can search over them.
- In this notebook, FAISS stores the chunk embeddings and performs nearest-neighbor search.

A common similarity measure is cosine similarity:

$$
\cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|}
$$

This notebook uses FAISS with L2 distance, which searches for vectors that are closest in Euclidean space.

In [ ]:
# Function to search for similar chunks
def search_similar_chunks(query, k=5):
    try:
        query_embedding = np.array(get_embedding(query)).astype("float32")
        query_embedding = query_embedding.reshape(1, -1)
        print(f"\nQuery embedding dim: {query_embedding.shape}")

        distances, indices = index.search(query_embedding, k)
        print("Distances:", distances)
        print("Indices:", indices)

        if indices.size == 0:
            print("No similar chunks found.")
            return []

        return [all_chunks[i] for i in indices[0]]
    except Exception as e:
        print(f"An error occurred: {e}")
        return []

# Function to load embeddings and chunks from the specified directory
def load_embeddings_and_chunks(pdf_directory):
    embedding_file_path = Path(pdf_directory) / "all_embeddings.npy"
    chunks_file_path = Path(pdf_directory) / "all_chunks.txt"

    embedding_matrix = np.load(embedding_file_path)

    with open(chunks_file_path, "r", encoding="utf-8") as f:
        content = f.read()
        all_chunks = content.split("\n")

    all_chunks = [chunk for chunk in all_chunks if chunk.strip() != ""]
    return embedding_matrix, all_chunks

embedding_matrix, all_chunks = load_embeddings_and_chunks(pdf_directory)

index = faiss.IndexFlatL2(embedding_matrix.shape[1])
index.add(embedding_matrix)
print(f"\nFAISS index: {index.ntotal} embeddings loaded.")

query = "What does the supplementary material say about evaluating RAG systems or retrieval quality?"
matched_chunks = search_similar_chunks(query)

if matched_chunks:
    print(f"\nMost similar chunk:\n{matched_chunks[0]}")

    if len(matched_chunks) > 1:
        print("\nOther matched chunks:")
        for i, chunk in enumerate(matched_chunks[1:], start=1):
            print(f"\nChunk {i}:\n{chunk}\n")
else:
    print("No similar chunks found.")

# PART 2: Augmented Generation

Now that we can retrieve relevant chunks, we can pass them to the LLM and ask for an answer grounded in the retrieved context.

## From Retrieval to Generation

Retrieval alone gives us relevant chunks, but not a final answer. The generation step asks the LLM to read those chunks and produce a response grounded in them.

Conceptually, the workflow is:

$$
\text{answer} = \text{LLM}(\text{query},\; \text{retrieved context})
$$

The key idea is that the model does not answer from parametric memory alone. Instead, we inject external context at inference time.

This is useful when:
- the documents are private or domain-specific,
- the knowledge changes often,
- we want answers tied to source material rather than unsupported recall.

In [ ]:
def answer_query_with_rag(query, k=5):
    matched_chunks = search_similar_chunks(query, k=k)

    if not matched_chunks:
        return "No relevant chunks were found for this query."

    context = "\n\n".join(matched_chunks)

    prompt = f"""
    You are a helpful assistant answering questions using the provided document context.

    Context:
    {context}

    Question:
    {query}

    Instructions:
    - Answer only from the provided context.
    - If the answer is not in the context, say so clearly.
    - Provide a concise but informative explanation.
    """

    response = my_llm.invoke(prompt)
    return response.content if hasattr(response, "content") else str(response)

query = "Summarize the main ideas discussed in the RADIANT_LLM supplementary material."
rag_response = answer_query_with_rag(query, k=5)

display(Markdown("### User Query"))
display(Markdown(query))
display(Markdown("### RAG Response"))
display(Markdown(rag_response))

# PART 3: Wrap the RAG Pipeline as a Tool

In this section, we wrap the retrieval pipeline as a tool and connect it to a simple LangChain agent.

In [ ]:
@tool
def pdf_rag_tool(query: str, k: int = 5) -> str:
    """
    Search the downloaded PDF collection and answer the question using retrieved document context.
    """
    return answer_query_with_rag(query, k=k)






tools = [
    pdf_rag_tool,
]

## Initialize the Agent

## What the Agent Is Doing

- A plain RAG pipeline always follows the same fixed retrieval-then-generation sequence.
- An agent adds a decision layer: it can choose when to call the PDF RAG tool and then incorporate the result into its final response.
- For this public tutorial, the goal is not broad autonomy. The goal is to show how a RAG pipeline can be wrapped as a reusable tool.

In [ ]:
memory = ConversationBufferMemory(
    return_messages=True,
    memory_key="chat_history",
    input_key="input",
)

prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful expert assistant. Use the tools when needed. "
        "If the answer is not available from the tools or the retrieved context, do not make it up."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(my_llm, tools, prompt_template)
rag_doc_agent = AgentExecutor(
    agent=agent,
    tools=tools,
    handle_parsing_errors=True,
    verbose=False,
    memory=memory,
)

## Query the Agent

In [ ]:
query = (
    "According to the repository supplementary material, what themes or evaluation goals are emphasized? "
    "Give a short answer grounded in the document context."
)

response = rag_doc_agent.invoke({"input": query})

display(Markdown("### User Query"))
display(Markdown(query))
display(Markdown("### Agent Response"))
display(Markdown(response["output"]))

## Suggested Follow-Up Questions

- What happens if we reduce the chunk size?
- What happens if we increase the number of retrieved chunks `k`?
- When is a plain LLM enough, and when do we need RAG?
- Why can retrieved context improve trustworthiness for project-specific questions?
- What changes when we switch the model used for answer generation?
- How does changing `temperature` affect the style and consistency of answers?

## Suggested Next Step

Build a small RAG workflow around a document collection you care about, then compare its behavior against a plain LLM without retrieval.

A useful extension is to swap in additional `RADIANT_LLM` materials, compare chunking choices, and inspect how retrieval quality changes before and after prompt or document updates.